# 12 — Generate Falcon 9 paper figures

This is the canonical figure-production notebook for the Falcon 9 Fireball paper.

It reads finalized geometry, waveform, measurement, and array-analysis products
created by earlier notebooks and generates publication figures without silently
recomputing scientific measurements.

Current figure groups:

1. SLC-40 and BCHH deployment geometry;
2. initial second-stage failure waveforms;
3. principal explosion waveforms;
4. capsule-related acoustic pulses;
5. catalogue array-result summary.

Additional final and supplementary figures can be added here as their upstream
products are frozen.

**Required upstream products**

- standardized geometry products from Notebook 02;
- corrected BCHH analysis waveform;
- key-event pressure measurements;
- event-catalogue array results.


I augmented the merged notebook with the main missing figure workflows recovered from the obsolete notebooks:
30-minute six-channel BCHH overview
observed-time initial-sequence chronology
reduced-time chronology with source-time overlays
optional synchronized consumer audio
reduced-time close-ups for second stage, first stage, and capsule
helicorder-style supplementary overview
expanded figure manifest
Download the augmented figure notebook
A few settings will need checking locally before execution, especially:
AUDIO_FILE
AUDIO_STARTTIME
AUDIO_CLOCK_SHIFT_S
AUDIO_REDUCTION_DELAY_S
SEISMIC_REDUCTION_MODE
The last setting supports either direct-seismic reduction or acoustic reduction for the HH channels, since the correct choice depends on whether the figure is emphasizing the early direct seismic onset or the dominant ground-coupled airwave.
I removed the old hidden timing tweaks rather than carrying them into the production notebook. The notebook structure is valid, but I could not execute it against your local waveform, geometry, module, and audio files

## 1. Imports, project paths, and plotting configuration


In [2]:

from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import CRS, Geod, Transformer

from obspy import Stream, UTCDateTime, read

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import ensure_output_dirs
from geometry_products import read_geometry_products
from figure1_utils import (
    get_sensor_location,
    make_figure1,
    save_figure,
)
from plotting import plot_key_event_waveforms

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

WGS84 = CRS.from_epsg(4326)
UTM17N = CRS.from_epsg(32617)

geod = Geod(ellps="WGS84")
ll_to_utm = Transformer.from_crs(WGS84, UTM17N, always_xy=True)
utm_to_ll = Transformer.from_crs(UTM17N, WGS84, always_xy=True)

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Derived data: {DERIVED_DIR}")
print(f"Figure output: {FIGURE_DIR}")


Project root: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper
Derived data: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper/outputs/derived
Figure output: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper/outputs/figures


## 2. Figure 1 — SLC-40 and BCHH deployment geometry

This section consumes the standardized geometry products written by Notebook 02.
Coordinates are not re-entered manually.


In [3]:
(
    inventory_event,
    channels_df,
    stations_df,
    locations_df,
) = read_geometry_products(DERIVED_DIR)

print("Channel columns:")
print(channels_df.columns.tolist())

print("\nLocation columns:")
print(locations_df.columns.tolist())

display(stations_df)
display(channels_df)
display(locations_df)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper/outputs/derived/event_inventory.xml'

### 2.1 Adapt the normalized channel table for mapping

The three seismic components represent one physical seismometer location and are
collapsed to a single `Seismometer` row.


In [ ]:
CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "HHZ": "Seismometer",
    "HHN": "Seismometer",
    "HHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "HHZ": "BCHH",
    "HHN": "BCHH",
    "HHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

required_channel_columns = {
    "channel",
    "latitude",
    "longitude",
}
missing = required_channel_columns.difference(channels_df.columns)
if missing:
    raise KeyError(
        "The standardized channel table is missing required columns: "
        f"{sorted(missing)}"
    )

channels_for_map = channels_df.copy()
channel_codes = (
    channels_for_map["channel"]
    .astype(str)
    .str.upper()
)

channels_for_map["sensor"] = channel_codes.map(CHANNEL_TO_SENSOR)
channels_for_map["label"] = channel_codes.map(CHANNEL_TO_LABEL)
channels_for_map["lat"] = channels_for_map["latitude"].astype(float)
channels_for_map["lon"] = channels_for_map["longitude"].astype(float)

eastings, northings = ll_to_utm.transform(
    channels_for_map["lon"].to_numpy(),
    channels_for_map["lat"].to_numpy(),
)
channels_for_map["easting"] = eastings
channels_for_map["northing"] = northings

unmapped = channels_for_map.loc[
    channels_for_map["sensor"].isna(),
    ["seed_id", "channel", "sensor_description"],
]
if not unmapped.empty:
    print("Ignoring channels that are not used in Figure 1:")
    display(unmapped)

mapped = channels_for_map.dropna(subset=["sensor"]).copy()

seismometer_rows = mapped.loc[
    mapped["sensor"] == "Seismometer"
]
if seismometer_rows.empty:
    raise ValueError("No seismic component was mapped to 'Seismometer'.")

# All three components are co-located; retain one representative row.
seismometer_row = seismometer_rows.iloc[[0]].copy()

infrasound_rows = (
    mapped.loc[mapped["sensor"].isin(["HD1", "HD2", "HD3"])]
    .sort_values("sensor")
    .drop_duplicates(subset=["sensor"])
)

bchh_sensors_df = pd.concat(
    [seismometer_row, infrasound_rows],
    ignore_index=True,
)

expected_sensors = {"Seismometer", "HD1", "HD2", "HD3"}
actual_sensors = set(bchh_sensors_df["sensor"])
if actual_sensors != expected_sensors:
    raise ValueError(
        "Expected exactly these physical sensors: "
        f"{sorted(expected_sensors)}; found {sorted(actual_sensors)}"
    )

figure1_columns = [
    "sensor",
    "label",
    "lat",
    "lon",
    "easting",
    "northing",
    "channel",
    "seed_id",
    "sensor_description",
]
display(bchh_sensors_df[figure1_columns])


### 2.2 Extract launch-pad and BCHH reference locations


In [ ]:
def get_unique_kml_location(
    dataframe: pd.DataFrame,
    kml_id: str,
) -> dict:
    matches = dataframe.loc[
        dataframe["kml_id"]
        .astype(str)
        .str.upper()
        .eq(kml_id.upper())
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one KML placemark {kml_id!r}; "
            f"found {len(matches)}."
        )

    row = matches.iloc[0].copy()

    # Normalize possible coordinate-column conventions.
    lat_key = (
        "lat"
        if "lat" in row.index
        else "latitude"
    )
    lon_key = (
        "lon"
        if "lon" in row.index
        else "longitude"
    )

    return {
        **row.to_dict(),
        "name": str(row.get("name", kml_id)),
        "lat": float(row[lat_key]),
        "lon": float(row[lon_key]),
    }


SLC40 = get_unique_kml_location(locations_df, "SLC40")
SLC41 = get_unique_kml_location(locations_df, "SLC41")

BCHH = get_sensor_location(
    bchh_sensors_df,
    sensor_name="Seismometer",
    display_name="BCHH",
)

print("SLC-40:", SLC40)
print("SLC-41:", SLC41)
print("BCHH:", BCHH)


### 2.3 Generate and save Figure 1


In [ ]:
fig, axes = make_figure1(
    slc40=SLC40,
    slc41=SLC41,
    bchh=BCHH,
    bchh_sensors=bchh_sensors_df,
    geod=geod,
    ll_to_utm=ll_to_utm,
    utm_to_ll=utm_to_ll,
)

output_paths = save_figure(
    fig=fig,
    output_directory=FIGURE_DIR,
    filename_stem="fig01_slc40_bchh_location",
    extensions=("png", "pdf"),
    dpi=300,
)

print("Saved Figure 1:")
for path in output_paths:
    print(f"  {Path(path).resolve()}")

plt.show()


## 3. Load finalized waveform and key-event measurement products

The figure notebook reads measurements from CSV and does not recompute them.


In [ ]:

CORRECTED_WAVEFORM_FILE = DERIVED_DIR / "bchh_corrected_analysis_window.pkl"
KEY_EVENT_MEASUREMENTS_FILE = DERIVED_DIR / "key_event_pressure_measurements.csv"

for required_path, label in [
    (CORRECTED_WAVEFORM_FILE, "corrected waveform"),
    (KEY_EVENT_MEASUREMENTS_FILE, "key-event measurements"),
]:
    if not required_path.exists():
        raise FileNotFoundError(f"{label} not found: {required_path}")

st_corr = read(str(CORRECTED_WAVEFORM_FILE), format="PICKLE")
measurements = pd.read_csv(KEY_EVENT_MEASUREMENTS_FILE)

print(st_corr)
display(measurements)


## 4. Initial second-stage failure


In [ ]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")
t0, t1 = EXPLOSION_TIME + 3.0 - 0.08, EXPLOSION_TIME + 5.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Initial second-stage failure"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Initial second-stage failure",
    outfile=FIGURE_DIR / "second_stage_waveforms.png",
)
plt.show()


## 5. Principal explosion


In [ ]:

t0, t1 = EXPLOSION_TIME + 6.0 - 0.08, EXPLOSION_TIME + 9.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Principal explosion"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Principal Falcon 9 explosion",
    outfile=FIGURE_DIR / "principal_explosion_waveforms.png",
)
plt.show()


## 6. Capsule-related acoustic pulses


In [ ]:

t0 = UTCDateTime("2016-09-01T13:07:27.0")
t1 = UTCDateTime("2016-09-01T13:07:30.0")
st_event = st_corr.copy().trim(t0, t1)
capsule_results = measurements.loc[
    measurements["event"].isin(["Capsule pulse 1", "Capsule pulse 2"])
]
# For a two-pulse figure, marker annotations should be added manually or by
# extending plotting.py to accept multiple measurement windows.
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=None,
    title="Capsule-related acoustic pulses",
    outfile=FIGURE_DIR / "capsule_waveforms.png",
    add_measurements=False,
)
plt.show()


## 7. Shared chronology-figure settings and helpers

The next sections restore the principal figures developed in the older figure
notebooks:

- a 30-minute overview of the complete explosion sequence;
- an observed-time audio/seismic/infrasound chronology;
- a reduced-time chronology aligned to interpreted source times;
- optional close-ups of the second-stage, first-stage, and capsule onsets;
- an optional helicorder-style supplementary overview.

All travel-time, filter, and audio-synchronization choices are explicit settings
rather than hidden manual adjustments.


In [ ]:

import matplotlib.dates as mdates
from matplotlib.patches import Patch

# ---------------------------------------------------------------------
# Manuscript timing and propagation settings
# ---------------------------------------------------------------------
EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.02")
FIRST_STAGE = UTCDateTime("2016-09-01T13:07:15.000") + 22 / 30
CAPSULE = UTCDateTime("2016-09-01T13:07:24.000") + 16 / 30

SOURCE_EVENTS = {
    "Second\nstage": EXPLOSION_TIME,
    "First\nstage": FIRST_STAGE,
    "Capsule": CAPSULE,
}

PHASES = [
    ("Phase I", EXPLOSION_TIME, EXPLOSION_TIME + 70),
    ("Phase II", UTCDateTime("2016-09-01T13:14:15"), UTCDateTime("2016-09-01T13:15:45")),
    ("Phase III", UTCDateTime("2016-09-01T13:19:12"), UTCDateTime("2016-09-01T13:25:00")),
    ("Phase IV", UTCDateTime("2016-09-01T13:29:54"), UTCDateTime("2016-09-01T13:34:54")),
]

TREMOR_START = UTCDateTime("2016-09-01T13:08:48")
TREMOR_END = UTCDateTime("2016-09-01T13:12:00")

ACOUSTIC_SPEED_MPS = 350.4
SEISMIC_SPEED_MPS = 1100.0

SENSOR_DISTANCES_M = {
    "HHE": 1421.8,
    "HHN": 1421.8,
    "HHZ": 1421.8,
    "HD1": 1442.2,
    "HD2": 1410.2,
    "HD3": 1414.8,
}

# "seismic" uses distance / SEISMIC_SPEED_MPS for HH channels.
# "acoustic" uses distance / ACOUSTIC_SPEED_MPS for HH channels and is useful
# when plotting the dominant ground-coupled airwave rather than an early direct
# seismic arrival.
SEISMIC_REDUCTION_MODE = "seismic"

CHANNEL_ORDER = ["AUD", "HD1", "HD2", "HD3", "HHE", "HHN", "HHZ"]

# Optional synchronized consumer-audio file. Edit as required.
AUDIO_FILE_CANDIDATES = [
    PROJECT_ROOT / "data" / "public_video" / "syncedsound_14s.mp3",
    Path(
        "/Users/thompsong/Library/CloudStorage/Box-Box/thompsong/"
        "3_Project_Documents/NASAprojects/201602_Rocket_Seismology/"
        "02_KSC/spaceX_explosion_movie/syncedsound_14s.mp3"
    ),
]
AUDIO_FILE = next((path for path in AUDIO_FILE_CANDIDATES if path.exists()), None)

# Absolute UTC assigned to sample zero of the synchronized audio.
AUDIO_STARTTIME = UTCDateTime("2016-09-01T13:07:10.000")

# Small synchronization correction applied after assigning AUDIO_STARTTIME.
AUDIO_CLOCK_SHIFT_S = 0.10

# Propagation delay to remove in the reduced-time figure.
# Keep at 0 if the synchronized audio has already been shifted to source time.
AUDIO_REDUCTION_DELAY_S = 0.0

# Display filters. These affect figures only, not scientific measurements.
OVERVIEW_INFRA_FILTER = {"freqmin": 0.5, "freqmax": 25.0, "corners": 4, "zerophase": True}
OVERVIEW_SEIS_FILTER = {"freqmin": 1.0, "freqmax": 40.0, "corners": 4, "zerophase": True}

ZOOM_INFRA_FILTER = {"freqmin": 0.5, "freqmax": 40.0, "corners": 4, "zerophase": True}
ZOOM_SEIS_FILTER = {"freqmin": 2.0, "freqmax": 60.0, "corners": 4, "zerophase": True}
ZOOM_AUDIO_FILTER = {"freqmin": 1000.0, "freqmax": 10000.0, "corners": 4, "zerophase": True}

EVENT_WINDOW_S = 0.10


def is_audio_channel(channel: str) -> bool:
    return str(channel).upper() in {"AUD", "AUDIO"}


def is_infrasound_channel(channel: str) -> bool:
    return str(channel).upper().startswith(("HD", "DD"))


def is_seismic_channel(channel: str) -> bool:
    return str(channel).upper().startswith(("HH", "DH"))


def robust_scale(data, percentile: float = 99.5) -> float:
    values = np.asarray(data, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 1.0
    scale = np.percentile(np.abs(values), percentile)
    return float(scale) if np.isfinite(scale) and scale > 0 else 1.0


def trace_time_array_datetime64(trace):
    start = np.datetime64(trace.stats.starttime.datetime)
    dt_ns = int(round(trace.stats.delta * 1e9))
    return start + np.arange(trace.stats.npts) * np.timedelta64(dt_ns, "ns")


def order_stream(stream: Stream, channel_order=CHANNEL_ORDER) -> Stream:
    ordered = Stream()
    for channel in channel_order:
        matches = stream.select(channel=channel)
        if len(matches):
            ordered += matches[0].copy()
    return ordered


def filter_display_stream(
    stream: Stream,
    starttime: UTCDateTime,
    endtime: UTCDateTime,
    infra_filter: dict | None = None,
    seis_filter: dict | None = None,
    pad_seconds: float = 10.0,
) -> Stream:
    """Trim and filter a display copy without altering the scientific products."""
    result = Stream()

    for trace in stream:
        tr = trace.copy()
        tr.trim(
            starttime - pad_seconds,
            endtime + pad_seconds,
            pad=False,
        )
        tr.detrend("demean")
        tr.detrend("linear")
        tr.taper(max_percentage=0.02)

        settings = (
            infra_filter if is_infrasound_channel(tr.stats.channel)
            else seis_filter if is_seismic_channel(tr.stats.channel)
            else None
        )

        if settings:
            nyquist = 0.5 * tr.stats.sampling_rate
            freqmax = min(float(settings["freqmax"]), 0.95 * nyquist)
            freqmin = float(settings["freqmin"])
            if freqmin < freqmax:
                tr.filter(
                    "bandpass",
                    freqmin=freqmin,
                    freqmax=freqmax,
                    corners=int(settings.get("corners", 4)),
                    zerophase=bool(settings.get("zerophase", True)),
                )

        tr.trim(starttime, endtime, pad=False)
        result += tr

    return order_stream(result)


def save_png_pdf(fig, outfile_base: Path, dpi: int = 300):
    outfile_base.parent.mkdir(parents=True, exist_ok=True)
    output_paths = []
    for extension in ("png", "pdf"):
        path = outfile_base.with_suffix(f".{extension}")
        fig.savefig(path, dpi=dpi, bbox_inches="tight", pad_inches=0.04)
        output_paths.append(path)
        print(f"Saved: {path}")
    return output_paths


## 8. Thirty-minute overview of the explosion sequence

This restores the broad overview from the obsolete notebooks. It shows all six
BCHH channels, the four phases of impulsive activity, and the interval of continuous
seismic tremor.


In [ ]:

def add_time_bar(ax, starttime, endtime, y, text, above=True, lw=1.4, fontsize=9):
    x0 = mdates.date2num(starttime.datetime)
    x1 = mdates.date2num(endtime.datetime)

    ax.plot([x0, x1], [y, y], color="black", lw=lw, clip_on=False, zorder=20)
    tick = 0.08
    ax.plot([x0, x0], [y - tick, y + tick], color="black", lw=lw, clip_on=False, zorder=20)
    ax.plot([x1, x1], [y - tick, y + tick], color="black", lw=lw, clip_on=False, zorder=20)

    dy = 0.12 if above else -0.12
    ax.text(
        (x0 + x1) / 2,
        y + dy,
        text,
        ha="center",
        va="bottom" if above else "top",
        fontsize=fontsize,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 1.0},
        clip_on=False,
        zorder=25,
    )


def make_overview_figure(
    stream: Stream,
    outfile_base: Path,
    plot_start: UTCDateTime,
    plot_end: UTCDateTime,
    phases: list,
    tremor_start: UTCDateTime | None = None,
    tremor_end: UTCDateTime | None = None,
    normalize_by_group: bool = False,
    clip: float = 3.0,
    trace_gain: float = 0.16,
):
    if len(stream) == 0:
        raise ValueError("No traces to plot")

    st_plot = order_stream(stream)
    fig, ax = plt.subplots(figsize=(11.0, 5.8))

    offsets = np.arange(len(st_plot))[::-1].astype(float)
    labels = []

    infra_scales = [
        robust_scale(tr.data, 99.3)
        for tr in st_plot if is_infrasound_channel(tr.stats.channel)
    ]
    seis_scales = [
        robust_scale(tr.data, 99.3)
        for tr in st_plot if is_seismic_channel(tr.stats.channel)
    ]
    infra_group_scale = np.median(infra_scales) if infra_scales else 1.0
    seis_group_scale = np.median(seis_scales) if seis_scales else 1.0

    for index, tr in enumerate(st_plot):
        y0 = offsets[index]
        channel = tr.stats.channel.upper()
        data = np.nan_to_num(tr.data.astype(float))

        if normalize_by_group:
            scale = infra_group_scale if is_infrasound_channel(channel) else seis_group_scale
        else:
            scale = robust_scale(data, 99.3)

        y = y0 + np.clip(data / scale, -clip, clip) * trace_gain
        ax.plot(
            trace_time_array_datetime64(tr),
            y,
            color="black",
            lw=0.45,
            rasterized=True,
        )
        labels.append(channel)

    ax.set_yticks(offsets)
    ax.set_yticklabels(labels)
    ax.set_ylim(-1.05, len(st_plot) + 0.55)
    ax.set_xlim(
        mdates.date2num(plot_start.datetime),
        mdates.date2num(plot_end.datetime),
    )

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_minor_locator(mdates.MinuteLocator(interval=1))
    ax.grid(axis="x", which="major", color="0.78", lw=0.8)
    ax.grid(axis="x", which="minor", color="0.93", lw=0.4)
    ax.set_xlabel("Time on 1 September 2016 (UTC)")

    # For the default order HD1, HD2, HD3, HHE, HHN, HHZ.
    if len(st_plot) >= 6:
        ax.axhline(2.5, color="0.55", lw=0.8)

    phase_y = len(st_plot) + 0.18
    for name, start, end in phases:
        if end < plot_start or start > plot_end:
            continue
        add_time_bar(
            ax,
            max(start, plot_start),
            min(end, plot_end),
            phase_y,
            name,
            above=True,
        )

    if tremor_start is not None and tremor_end is not None:
        if tremor_end >= plot_start and tremor_start <= plot_end:
            add_time_bar(
                ax,
                max(tremor_start, plot_start),
                min(tremor_end, plot_end),
                y=-0.4,
                text="continuous seismic tremor",
                above=False,
                lw=1.2,
            )

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    fig.subplots_adjust(left=0.08, right=0.99, top=0.88, bottom=0.14)
    save_png_pdf(fig, outfile_base)
    return fig, ax


OVERVIEW_START = EXPLOSION_TIME - 60
OVERVIEW_END = EXPLOSION_TIME + 30 * 60

st_overview = filter_display_stream(
    st_corr,
    starttime=OVERVIEW_START,
    endtime=OVERVIEW_END,
    infra_filter=OVERVIEW_INFRA_FILTER,
    seis_filter=OVERVIEW_SEIS_FILTER,
    pad_seconds=20.0,
)

fig, ax = make_overview_figure(
    st_overview,
    FIGURE_DIR / "fig03_bchh_30min_overview",
    plot_start=OVERVIEW_START,
    plot_end=OVERVIEW_END,
    phases=PHASES,
    tremor_start=TREMOR_START,
    tremor_end=TREMOR_END,
)
plt.show()


## 9. Optional synchronized consumer audio

The audio is an uncalibrated timing record. It is useful for relative timing but
must not be used for physical amplitude, energy, or source-spectrum estimates.

The section is skipped cleanly when the audio file is unavailable.


In [ ]:

def load_video_audio_as_stream(
    audio_file: Path,
    starttime: UTCDateTime,
    clock_shift_s: float = 0.0,
) -> Stream:
    try:
        import librosa
    except ImportError as exc:
        raise ImportError(
            "Install librosa to load the synchronized video audio."
        ) from exc

    samples, sampling_rate = librosa.load(audio_file, sr=None, mono=True)

    from obspy import Trace

    trace = Trace(data=np.asarray(samples, dtype=np.float32))
    trace.stats.sampling_rate = float(sampling_rate)
    trace.stats.starttime = starttime + clock_shift_s
    trace.stats.network = "YT"
    trace.stats.station = "AUDIO"
    trace.stats.location = ""
    trace.stats.channel = "AUD"

    return Stream([trace])


def prepare_audio_for_display(
    stream: Stream,
    starttime: UTCDateTime,
    endtime: UTCDateTime,
    filter_settings: dict,
    pad_seconds: float = 10.0,
) -> Stream:
    result = stream.copy()

    for tr in result:
        tr.trim(starttime - pad_seconds, endtime + pad_seconds, pad=False)
        tr.detrend("demean")
        tr.taper(max_percentage=0.02)

        nyquist = 0.5 * tr.stats.sampling_rate
        freqmax = min(float(filter_settings["freqmax"]), 0.95 * nyquist)
        freqmin = float(filter_settings["freqmin"])

        if freqmin < freqmax:
            tr.filter(
                "bandpass",
                freqmin=freqmin,
                freqmax=freqmax,
                corners=int(filter_settings.get("corners", 4)),
                zerophase=bool(filter_settings.get("zerophase", True)),
            )

        tr.trim(starttime, endtime, pad=False)
        tr.normalize()

    return result


if AUDIO_FILE is None:
    st_audio_raw = Stream()
    print(
        "Synchronized audio not found. Set AUDIO_FILE in the settings cell "
        "to enable audio chronology figures."
    )
else:
    st_audio_raw = load_video_audio_as_stream(
        AUDIO_FILE,
        starttime=AUDIO_STARTTIME,
        clock_shift_s=AUDIO_CLOCK_SHIFT_S,
    )
    print(f"Loaded audio: {AUDIO_FILE}")
    print(st_audio_raw)


## 10. Observed-time and reduced-time chronology

Two complementary versions are generated:

- **observed time:** raw timing relationships, with no interpretive overlays;
- **reduced time:** each trace is shifted earlier by its specified propagation
  delay, and interpreted source times are shown as red lines with 0.1 s grey
  timing windows.

The HH-channel reduction can use either direct-seismic or acoustic travel time,
controlled by `SEISMIC_REDUCTION_MODE`.


In [ ]:

def channel_reduction_delay_s(
    channel: str,
    sensor_distances_m: dict,
    acoustic_speed_mps: float,
    seismic_speed_mps: float,
    audio_delay_s: float,
    seismic_mode: str = "seismic",
) -> float:
    channel = str(channel).upper()

    if is_audio_channel(channel):
        return float(audio_delay_s)

    if channel not in sensor_distances_m:
        return 0.0

    distance = float(sensor_distances_m[channel])

    if is_infrasound_channel(channel):
        return distance / acoustic_speed_mps

    if is_seismic_channel(channel):
        if seismic_mode == "acoustic":
            return distance / acoustic_speed_mps
        if seismic_mode == "seismic":
            return distance / seismic_speed_mps
        raise ValueError("seismic_mode must be 'seismic' or 'acoustic'")

    return 0.0


def reduce_stream_to_source_time(
    stream: Stream,
    sensor_distances_m: dict,
    acoustic_speed_mps: float,
    seismic_speed_mps: float,
    audio_delay_s: float = 0.0,
    seismic_mode: str = "seismic",
) -> Stream:
    reduced = stream.copy()

    for tr in reduced:
        delay = channel_reduction_delay_s(
            tr.stats.channel,
            sensor_distances_m=sensor_distances_m,
            acoustic_speed_mps=acoustic_speed_mps,
            seismic_speed_mps=seismic_speed_mps,
            audio_delay_s=audio_delay_s,
            seismic_mode=seismic_mode,
        )
        tr.stats.starttime -= delay
        print(f"Reduced {tr.id} by {delay:.4f} s")

    return reduced


def make_chronology_figure(
    stream: Stream,
    outfile_base: Path,
    plot_start: UTCDateTime,
    plot_end: UTCDateTime,
    source_events: dict,
    reduced_time: bool,
    show_source_times: bool,
    show_event_windows: bool,
    event_window_s: float = 0.10,
    clip: float = 1.2,
    trace_gain: float = 0.38,
):
    if len(stream) == 0:
        raise ValueError("No traces to plot")

    st_plot = order_stream(stream)
    fig, ax = plt.subplots(figsize=(11.0, 5.4))

    offsets = np.arange(len(st_plot))[::-1].astype(float)
    labels = []

    for index, tr in enumerate(st_plot):
        y0 = offsets[index]
        channel = tr.stats.channel.upper()
        trc = tr.copy().trim(plot_start, plot_end, pad=True, fill_value=0)
        data = np.nan_to_num(trc.data.astype(float))
        scale = robust_scale(data, 99.5)

        y = y0 + np.clip(data / scale, -clip, clip) * trace_gain
        ax.plot(
            trace_time_array_datetime64(trc),
            y,
            color="black",
            lw=0.55,
            rasterized=True,
            zorder=12,
        )

        labels.append(
            "Consumer audio\n1–10 kHz" if is_audio_channel(channel) else channel
        )

    ax.set_yticks(offsets)
    ax.set_yticklabels(labels)
    ax.set_ylim(-0.7, len(st_plot) - 0.25)
    ax.set_xlim(
        mdates.date2num(plot_start.datetime),
        mdates.date2num(plot_end.datetime),
    )

    ax.xaxis.set_major_locator(mdates.SecondLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.xaxis.set_minor_locator(mdates.SecondLocator(interval=1))
    ax.grid(axis="x", which="major", color="0.78", lw=0.8, zorder=1)
    ax.grid(axis="x", which="minor", color="0.93", lw=0.4, zorder=1)

    ax.set_xlabel(
        "Reduced time on 1 September 2016 (UTC equivalent)"
        if reduced_time
        else "Observed time on 1 September 2016 (UTC)"
    )

    # Draw separators based on the actual ordered trace list.
    audio_positions = [
        offsets[i] for i, tr in enumerate(st_plot)
        if is_audio_channel(tr.stats.channel)
    ]
    infra_positions = [
        offsets[i] for i, tr in enumerate(st_plot)
        if is_infrasound_channel(tr.stats.channel)
    ]

    if audio_positions and infra_positions:
        ax.axhline((min(audio_positions) + max(infra_positions)) / 2, color="0.55", lw=0.8)

    seis_positions = [
        offsets[i] for i, tr in enumerate(st_plot)
        if is_seismic_channel(tr.stats.channel)
    ]
    if infra_positions and seis_positions:
        ax.axhline((min(infra_positions) + max(seis_positions)) / 2, color="0.55", lw=0.8)

    ytop = len(st_plot) - 0.15

    if reduced_time:
        for label, source_time in source_events.items():
            if show_event_windows:
                left = source_time - event_window_s / 2
                right = source_time + event_window_s / 2
                if right >= plot_start and left <= plot_end:
                    ax.axvspan(
                        max(left, plot_start).datetime,
                        min(right, plot_end).datetime,
                        color="0.75",
                        alpha=0.35,
                        lw=0,
                        zorder=18,
                    )

            if show_source_times and plot_start <= source_time <= plot_end:
                x = mdates.date2num(source_time.datetime)
                ax.axvline(x, color="firebrick", lw=1.1, alpha=0.85, zorder=25)
                ax.text(
                    x,
                    ytop,
                    label,
                    rotation=90,
                    ha="center",
                    va="top",
                    fontsize=8,
                    color="firebrick",
                    bbox={
                        "facecolor": "white",
                        "edgecolor": "none",
                        "alpha": 0.75,
                        "pad": 1.0,
                    },
                    clip_on=False,
                    zorder=30,
                )

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    fig.subplots_adjust(left=0.10, right=0.99, top=0.96, bottom=0.15)
    save_png_pdf(fig, outfile_base)
    return fig, ax


CHRONOLOGY_START = EXPLOSION_TIME - 2
CHRONOLOGY_END = EXPLOSION_TIME + 28

st_chronology_bchh = filter_display_stream(
    st_corr,
    starttime=CHRONOLOGY_START,
    endtime=CHRONOLOGY_END,
    infra_filter=ZOOM_INFRA_FILTER,
    seis_filter=ZOOM_SEIS_FILTER,
    pad_seconds=10.0,
)

st_chronology = st_chronology_bchh.copy()

if len(st_audio_raw):
    st_audio_display = prepare_audio_for_display(
        st_audio_raw,
        starttime=CHRONOLOGY_START,
        endtime=CHRONOLOGY_END,
        filter_settings=ZOOM_AUDIO_FILTER,
        pad_seconds=10.0,
    )
    st_chronology = st_audio_display + st_chronology

st_chronology = order_stream(st_chronology)

# Observed-time companion: no interpretive overlays.
fig, ax = make_chronology_figure(
    st_chronology,
    FIGURE_DIR / "fig04_initial_sequence_observed_time",
    plot_start=CHRONOLOGY_START,
    plot_end=CHRONOLOGY_END,
    source_events=SOURCE_EVENTS,
    reduced_time=False,
    show_source_times=False,
    show_event_windows=False,
)
plt.show()

# Reduced-time figure: source-time markers and timing windows.
st_chronology_reduced = reduce_stream_to_source_time(
    st_chronology,
    sensor_distances_m=SENSOR_DISTANCES_M,
    acoustic_speed_mps=ACOUSTIC_SPEED_MPS,
    seismic_speed_mps=SEISMIC_SPEED_MPS,
    audio_delay_s=AUDIO_REDUCTION_DELAY_S,
    seismic_mode=SEISMIC_REDUCTION_MODE,
)

fig, ax = make_chronology_figure(
    st_chronology_reduced,
    FIGURE_DIR / "fig04_initial_sequence_reduced_time",
    plot_start=CHRONOLOGY_START,
    plot_end=CHRONOLOGY_END,
    source_events=SOURCE_EVENTS,
    reduced_time=True,
    show_source_times=True,
    show_event_windows=True,
    event_window_s=EVENT_WINDOW_S,
)
plt.show()


## 11. Optional onset close-ups

These are intended primarily for supplementary material. Each uses the same
reduced-time convention and synchronization settings as the main chronology.


In [ ]:

CLOSEUP_WINDOWS = {
    "second_stage": (EXPLOSION_TIME - 0.75, EXPLOSION_TIME + 1.25),
    "first_stage": (FIRST_STAGE - 0.75, FIRST_STAGE + 1.25),
    "capsule": (CAPSULE - 0.75, CAPSULE + 1.25),
}

for event_name, (start, end) in CLOSEUP_WINDOWS.items():
    fig, ax = make_chronology_figure(
        st_chronology_reduced,
        FIGURE_DIR / f"figS_{event_name}_onset_reduced_time",
        plot_start=start,
        plot_end=end,
        source_events=SOURCE_EVENTS,
        reduced_time=True,
        show_source_times=True,
        show_event_windows=True,
        event_window_s=EVENT_WINDOW_S,
        clip=1.2,
        trace_gain=0.38,
    )
    plt.show()


## 12. Optional helicorder-style supplementary overview

This restores the long-record visualization from the obsolete notebooks. It is
useful when the later low-amplitude phases are difficult to see in the conventional
30-minute overview.


In [ ]:

def make_helicorder_figure(
    trace,
    outfile_base: Path,
    starttime: UTCDateTime,
    endtime: UTCDateTime,
    line_length_s: float = 60.0,
    scale_percentile: float = 99.5,
):
    tr = trace.copy().trim(starttime, endtime, pad=True, fill_value=0)
    sampling_rate = tr.stats.sampling_rate
    samples_per_line = int(round(line_length_s * sampling_rate))
    number_of_lines = int(np.ceil(tr.stats.npts / samples_per_line))

    scale = robust_scale(tr.data, scale_percentile)
    fig, ax = plt.subplots(figsize=(11.0, max(6.0, number_of_lines * 0.28)))

    for line_index in range(number_of_lines):
        i0 = line_index * samples_per_line
        i1 = min((line_index + 1) * samples_per_line, tr.stats.npts)
        segment = np.asarray(tr.data[i0:i1], dtype=float)
        elapsed = np.arange(segment.size) / sampling_rate
        y0 = number_of_lines - 1 - line_index
        ax.plot(
            elapsed,
            y0 + np.clip(segment / scale, -3, 3) * 0.32,
            color="black",
            lw=0.45,
            rasterized=True,
        )

    line_times = [starttime + index * line_length_s for index in range(number_of_lines)]
    ax.set_yticks(np.arange(number_of_lines)[::-1])
    ax.set_yticklabels([time.strftime("%H:%M") for time in line_times])
    ax.set_xlim(0, line_length_s)
    ax.set_xlabel("Seconds within each minute")
    ax.set_ylabel("UTC")
    ax.set_title(f"Helicorder overview: {tr.id}")
    ax.grid(axis="x", color="0.9", lw=0.5)

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    save_png_pdf(fig, outfile_base)
    return fig, ax


# The vertical seismic component is used by default because it displays both the
# continuous tremor and the strongest ground-coupled arrivals.
helicorder_matches = st_corr.select(channel="HHZ")
if len(helicorder_matches):
    fig, ax = make_helicorder_figure(
        helicorder_matches[0],
        FIGURE_DIR / "figS_helicorder_bchh_hhz",
        starttime=EXPLOSION_TIME - 60,
        endtime=EXPLOSION_TIME + 35 * 60,
    )
    plt.show()
else:
    print("HHZ not found; helicorder figure skipped.")


## 13. Catalogue array-result summary

This section reads the finalized array-results table. If the expected result file
does not exist, the notebook reports which upstream analysis must be run.


In [ ]:

ARRAY_RESULTS_CANDIDATES = [
    DERIVED_DIR / "planar_array_results.csv",
    DERIVED_DIR / "event_catalogue_array_results.csv",
]

array_file = next((path for path in ARRAY_RESULTS_CANDIDATES if path.exists()), None)

if array_file is None:
    print(
        "No finalized array-results table found. Expected one of:\n  "
        + "\n  ".join(str(path) for path in ARRAY_RESULTS_CANDIDATES)
    )
else:
    array_results = pd.read_csv(array_file)
    print(f"Loaded array results: {array_file}")
    display(array_results.head())

    required_columns = {"back_azimuth_deg"}
    missing = required_columns.difference(array_results.columns)
    if missing:
        raise KeyError(
            f"Array-results table is missing required columns: {sorted(missing)}"
        )

    color_column = (
        "mean_abs_correlation"
        if "mean_abs_correlation" in array_results.columns
        else None
    )

    fig, ax = plt.subplots(figsize=(10, 4))

    if color_column is not None:
        scatter = ax.scatter(
            np.arange(len(array_results)),
            array_results["back_azimuth_deg"],
            c=array_results[color_column],
            s=22,
        )
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label("Mean absolute correlation")
    else:
        ax.scatter(
            np.arange(len(array_results)),
            array_results["back_azimuth_deg"],
            s=22,
        )

    ax.set_xlabel("Event number")
    ax.set_ylabel("Back azimuth (degrees)")
    ax.set_title("Source direction through the explosion sequence")
    ax.grid(True, alpha=0.25)

    for ext in ("png", "pdf"):
        outfile = FIGURE_DIR / f"catalogue_back_azimuth.{ext}"
        fig.savefig(outfile, dpi=300, bbox_inches="tight")
        print(f"Saved: {outfile}")

    plt.show()


## 14. Figure manifest

The manifest records the files expected from the currently implemented figure
sections. Additional figures should be added here as the manuscript figure set is
finalized.


In [ ]:

figure_manifest = pd.DataFrame([
    {
        "figure": "Figure 1",
        "description": "SLC-40 and BCHH deployment geometry",
        "png": FIGURE_DIR / "fig01_slc40_bchh_location.png",
        "pdf": FIGURE_DIR / "fig01_slc40_bchh_location.pdf",
    },
    {
        "figure": "Key event",
        "description": "Initial second-stage failure waveforms",
        "png": FIGURE_DIR / "second_stage_waveforms.png",
        "pdf": None,
    },
    {
        "figure": "Key event",
        "description": "Principal explosion waveforms",
        "png": FIGURE_DIR / "principal_explosion_waveforms.png",
        "pdf": None,
    },
    {
        "figure": "Key event",
        "description": "Capsule-related acoustic pulses",
        "png": FIGURE_DIR / "capsule_waveforms.png",
        "pdf": None,
    },
    {
        "figure": "Overview",
        "description": "Thirty-minute six-channel BCHH overview",
        "png": FIGURE_DIR / "fig03_bchh_30min_overview.png",
        "pdf": FIGURE_DIR / "fig03_bchh_30min_overview.pdf",
    },
    {
        "figure": "Chronology",
        "description": "Initial sequence in observed time",
        "png": FIGURE_DIR / "fig04_initial_sequence_observed_time.png",
        "pdf": FIGURE_DIR / "fig04_initial_sequence_observed_time.pdf",
    },
    {
        "figure": "Chronology",
        "description": "Initial sequence in reduced time",
        "png": FIGURE_DIR / "fig04_initial_sequence_reduced_time.png",
        "pdf": FIGURE_DIR / "fig04_initial_sequence_reduced_time.pdf",
    },
    {
        "figure": "Supplement",
        "description": "Second-stage onset close-up in reduced time",
        "png": FIGURE_DIR / "figS_second_stage_onset_reduced_time.png",
        "pdf": FIGURE_DIR / "figS_second_stage_onset_reduced_time.pdf",
    },
    {
        "figure": "Supplement",
        "description": "First-stage onset close-up in reduced time",
        "png": FIGURE_DIR / "figS_first_stage_onset_reduced_time.png",
        "pdf": FIGURE_DIR / "figS_first_stage_onset_reduced_time.pdf",
    },
    {
        "figure": "Supplement",
        "description": "Capsule onset close-up in reduced time",
        "png": FIGURE_DIR / "figS_capsule_onset_reduced_time.png",
        "pdf": FIGURE_DIR / "figS_capsule_onset_reduced_time.pdf",
    },
    {
        "figure": "Supplement",
        "description": "HHZ helicorder-style overview",
        "png": FIGURE_DIR / "figS_helicorder_bchh_hhz.png",
        "pdf": FIGURE_DIR / "figS_helicorder_bchh_hhz.pdf",
    },
    {
        "figure": "Array result",
        "description": "Catalogue back azimuth",
        "png": FIGURE_DIR / "catalogue_back_azimuth.png",
        "pdf": FIGURE_DIR / "catalogue_back_azimuth.pdf",
    },
])

for column in ("png", "pdf"):
    figure_manifest[f"{column}_exists"] = figure_manifest[column].map(
        lambda value: Path(value).exists() if value is not None else False
    )

display(figure_manifest)

manifest_file = FIGURE_DIR / "figure_manifest.csv"
figure_manifest.astype(str).to_csv(manifest_file, index=False)
print(f"Saved manifest: {manifest_file}")



## Outputs

The notebook now includes the principal products recovered from the obsolete figure
notebooks:

- Figure 1 deployment geometry;
- key-event waveform panels;
- 30-minute BCHH overview;
- observed-time chronology;
- reduced-time chronology with synchronized audio when available;
- reduced-time onset close-ups;
- helicorder-style supplementary overview;
- catalogue back-azimuth summary;
- figure manifest.

The catalogue dashboard, phase-summary figure, and final timing-residual product
remain to be added after their upstream tabular products are finalized.
